In [ ]:
# ============================================================================
# ML Final Project - Few-Shot Learning
# train.ipynb: 訓練模型並輸出 model.pth (整合全資料集重新訓練)
# ============================================================================

import subprocess
import sys
import warnings
warnings.filterwarnings('ignore')

# 自動安裝依賴套件
def install_if_needed(pkg: str):
    try:
        __import__(pkg.split("==")[0])
    except ModuleNotFoundError:
        print(f"[INFO] Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# 安裝必要套件
for lib in ["open_clip_torch", "timm", "torchinfo", "pillow", "tqdm"]:
    install_if_needed(lib)

# 導入套件
import os
import random
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchinfo import summary
from PIL import Image
import numpy as np
import timm
import open_clip

# ============================================================================
# 設定參數
# ============================================================================
IMG_SIZE = 224
VAL_RATIO = 0.2
BATCH_SIZE = 16  # 修正：從2改為16，提升訓練效率
CLIP_EPOCHS = 60
TIMM_EPOCHS = 60
LEARNING_RATE = 1e-4
CLIP_LR = 1e-4
SEED = 42
DATA_ROOT = "/content/train"

# ============================================================================
# 設定隨機種子確保可重現性
# ============================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

# ============================================================================
# 資料前處理和增強
# ============================================================================
def build_transforms(img_size):
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize(img_size + 32),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    return train_transform, val_transform

# ============================================================================
# 自定義資料集類別
# ============================================================================
class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

# ============================================================================
# 建立資料載入器
# ============================================================================
def build_loaders(root, img_size, val_ratio, batch_size, seed):
    rng = random.Random(seed)

    # 載入 ImageFolder 來獲得類別結構
    base_ds = ImageFolder(root)

    # 提取圖片路徑和標籤
    image_paths = [sample[0] for sample in base_ds.samples]
    labels = [sample[1] for sample in base_ds.samples]

    # 分層取樣：每個類別按相同比例分割
    class_to_indices = {}
    for idx, label in enumerate(labels):
        if label not in class_to_indices:
            class_to_indices[label] = []
        class_to_indices[label].append(idx)

    train_idx, val_idx = [], []
    for idxs in class_to_indices.values():
        rng.shuffle(idxs)
        split = int(len(idxs) * (1 - val_ratio))
        train_idx += idxs[:split]
        val_idx += idxs[split:]

    # 建立資料集
    train_transform, val_transform = build_transforms(img_size)

    train_paths = [image_paths[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_paths = [image_paths[i] for i in val_idx]
    val_labels = [labels[i] for i in val_idx]

    train_dataset = CustomDataset(train_paths, train_labels, train_transform)
    val_dataset = CustomDataset(val_paths, val_labels, val_transform)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )

    return train_loader, val_loader, base_ds.classes

# ============================================================================
# 新增：建立全資料集載入器
# ============================================================================
def build_full_dataset_loader(root, img_size, batch_size, seed):
    """使用全部300張圖片的資料載入器"""
    rng = random.Random(seed)

    base_ds = ImageFolder(root)
    image_paths = [sample[0] for sample in base_ds.samples]
    labels = [sample[1] for sample in base_ds.samples]

    # 使用全部資料，相同的transform
    train_transform, _ = build_transforms(img_size)
    full_dataset = CustomDataset(image_paths, labels, train_transform)

    full_loader = DataLoader(
        full_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True
    )

    return full_loader

# ============================================================================
# CLIP Fine-tuning 模型
# ============================================================================
class CLIPFineTuner(nn.Module):
    def __init__(self, clip_model, num_classes):
        super(CLIPFineTuner, self).__init__()
        self.clip_model = clip_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(clip_model.visual.output_dim, num_classes)

        # 凍結 CLIP 的大部分參數
        for param in self.clip_model.parameters():
            param.requires_grad = False

        # 只訓練最後幾層
        for name, param in self.clip_model.visual.named_parameters():
            if "ln_post" in name or "proj" in name:
                param.requires_grad = True

    def forward(self, x):
        with torch.no_grad():
            image_features = self.clip_model.encode_image(x)
        image_features = image_features.float()
        image_features = self.dropout(image_features)
        return self.classifier(image_features)

# ============================================================================
# Standard timm ConvNeXt 模型
# ============================================================================
def build_standard_timm_model(num_classes):
    model = timm.create_model("convnext_tiny", pretrained=True, num_classes=num_classes)

    # 標準策略：只訓練分類頭
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = False

    return model

# ============================================================================
# 早停機制（修改：記錄實際停止的epoch）
# ============================================================================
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.005, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_score = None
        self.counter = 0
        self.best_weights = None
        self.best_epoch = 0  # 新增：記錄最佳epoch
        self.current_epoch = 0  # 新增：追蹤當前epoch

    def __call__(self, val_score, model, epoch):  # 修改：加入epoch參數
        self.current_epoch = epoch

        if self.best_score is None:
            self.best_score = val_score
            self.best_epoch = epoch
            self.save_checkpoint(model)
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                if self.restore_best_weights:
                    model.load_state_dict(self.best_weights)
                return True
        else:
            self.best_score = val_score
            self.best_epoch = epoch
            self.counter = 0
            self.save_checkpoint(model)
        return False

    def save_checkpoint(self, model):
        self.best_weights = model.state_dict().copy()

# ============================================================================
# 通用訓練函數（修改：記錄實際停止epoch）
# ============================================================================
def train_model_with_early_stopping(model, train_loader, val_loader, epochs, lr, device, model_name, patience=10):
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)

    # 學習率調度 - 根據驗證準確率調整
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, verbose=True, min_lr=1e-6
    )

    # 早停機制
    early_stopping = EarlyStopping(patience=patience, min_delta=0.005)

    print(f"[INFO] Training {model_name} with early stopping (patience={patience})...")

    # 顯示模型參數資訊
    try:
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"[INFO] {model_name} Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
        print(f"[INFO] {model_name} Trainable parameters: {trainable_params_count:,} ({trainable_params_count/1e6:.1f}M)")

        if total_params > 1000e6:
            print(f"[WARNING] {model_name} has more than 1000M parameters!")
        else:
            print(f"[INFO] {model_name} parameter count is within requirement (<1000M)")

        # 使用 torchinfo 顯示模型結構 (作業要求)
        try:
            print(f"[INFO] {model_name} Model Summary:")
            summary(model, (1, 3, IMG_SIZE, IMG_SIZE), device=device)
        except Exception as e:
            print(f"[WARNING] Could not display detailed model summary: {e}")

    except Exception as e:
        print(f"[WARNING] Could not display {model_name} model summary: {e}")

    best_acc = 0.0
    train_start_time = time.time()

    for epoch in range(1, epochs + 1):
        epoch_start_time = time.time()

        # 訓練
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)

        # 驗證
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total
        train_loss = running_loss / len(train_loader.dataset)
        epoch_time = time.time() - epoch_start_time

        print(f"{model_name} Epoch {epoch:02d}/{epochs} | Loss: {train_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Time: {epoch_time:.1f}s")

        # 更新學習率
        scheduler.step(val_acc)

        # 早停檢查（傳入epoch）
        if early_stopping(val_acc, model, epoch):
            print(f"[INFO] Early stopping triggered at epoch {epoch} for {model_name}")
            break

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{model_name.lower().replace(' ', '_')}_model.pth")

    total_train_time = time.time() - train_start_time
    print(f"[INFO] {model_name} training completed in {total_train_time:.1f}s, best accuracy: {best_acc*100:.2f}%")
    print(f"[INFO] {model_name} best epoch: {early_stopping.best_epoch}")

    return model, best_acc, early_stopping.best_epoch  # 修改：返回最佳epoch

# ============================================================================
# 新增：全資料集訓練函數
# ============================================================================
def train_final_model_fixed_epochs(model, train_loader, epochs, lr, device, model_name):
    """使用固定epoch數在全資料集上訓練"""
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)

    # 使用相同的學習率調度策略
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True, min_lr=1e-6
    )

    print(f"[INFO] 最終訓練 {model_name}，使用全部300張圖片，固定 {epochs} epochs")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        print(f"Final {model_name} Epoch {epoch:02d}/{epochs} | Loss: {train_loss:.4f}")

        # 使用訓練損失調整學習率（沒有驗證集了）
        scheduler.step(train_loss)

    return model

# ============================================================================
# Ensemble權重優化（網格搜索）
# ============================================================================
def optimize_ensemble_weights_grid_search(models, model_names, val_loader, device):
    """使用網格搜索找到最佳ensemble權重"""

    print("[INFO] 收集各模型驗證集預測結果...")

    # 收集所有模型的預測結果
    all_predictions = []
    val_labels = []

    for model_idx, model in enumerate(models):
        model.eval()
        predictions = []
        labels_temp = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                predictions.append(torch.softmax(outputs, dim=1).cpu().numpy())

                # 只在第一個模型時收集標籤
                if model_idx == 0:
                    labels_temp.extend(labels.numpy())

        all_predictions.append(np.vstack(predictions))

        if model_idx == 0:
            val_labels = np.array(labels_temp)

    print(f"[INFO] 驗證集樣本數: {len(val_labels)}")

    # 網格搜索最佳權重（兩個模型）
    best_acc = 0
    best_weights = None

    print("[INFO] 搜索最佳ensemble權重...")

    # 兩個模型的權重搜索
    weight_options = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    for w1 in weight_options:
        w2 = 1.0 - w1

        # 計算加權預測
        ensemble_pred = w1 * all_predictions[0] + w2 * all_predictions[1]

        predicted_classes = np.argmax(ensemble_pred, axis=1)
        acc = np.mean(predicted_classes == val_labels)

        if acc > best_acc:
            best_acc = acc
            best_weights = [w1, w2]

    return best_weights, best_acc

# ============================================================================
# Ensemble模型類別
# ============================================================================
class OptimizedEnsemble(nn.Module):
    def __init__(self, clip_model, timm_model, weights):
        super().__init__()
        self.clip_model = clip_model
        self.timm_model = timm_model
        self.register_buffer('weights', torch.FloatTensor(weights))

    def forward(self, x):
        with torch.no_grad():
            clip_output = torch.softmax(self.clip_model(x), dim=1)
            timm_output = torch.softmax(self.timm_model(x), dim=1)

            # 加權平均
            ensemble_output = (self.weights[0] * clip_output +
                             self.weights[1] * timm_output)

        return ensemble_output

# ============================================================================
# 主要訓練流程
# ============================================================================
print(f"[INFO] Loading data from {DATA_ROOT}")

if not os.path.exists(DATA_ROOT):
    print(f"[ERROR] Data directory not found: {DATA_ROOT}")
    print("[INFO] Please ensure your training data is in /content/train/")
else:
    # 建立資料載入器
    train_loader, val_loader, class_names = build_loaders(
        DATA_ROOT, IMG_SIZE, VAL_RATIO, BATCH_SIZE, SEED
    )

    num_classes = len(class_names)
    print(f"[INFO] Found {num_classes} classes: {class_names[:5]}{'...' if num_classes > 5 else ''}")
    print(f"[INFO] Train samples: {len(train_loader.dataset)}, Val samples: {len(val_loader.dataset)}")

    # 儲存結果和epoch數
    results = {}
    best_epochs = {}  # 新增：記錄最佳epoch數

    print("\n" + "="*80)
    print("階段一：使用80%資料進行模型選擇和參數調優")
    print("="*80)

    print("\n" + "="*80)
    print("方法一：CLIP Fine-tuning (最佳化訓練)")
    print("="*80)

    # 建立CLIP模型
    clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
    clip_finetuner = CLIPFineTuner(clip_model, num_classes).to(device)

    clip_finetuner, clip_acc, clip_best_epoch = train_model_with_early_stopping(
        clip_finetuner, train_loader, val_loader, CLIP_EPOCHS, CLIP_LR, device, "CLIP Fine-tuning", patience=15
    )
    results["clip_finetuning"] = clip_acc
    best_epochs["clip_finetuning"] = clip_best_epoch

    print("\n" + "="*80)
    print("方法二：Standard timm ConvNeXt (最佳化訓練)")
    print("="*80)

    standard_timm_model = build_standard_timm_model(num_classes).to(device)

    standard_timm_model, standard_acc, timm_best_epoch = train_model_with_early_stopping(
        standard_timm_model, train_loader, val_loader, TIMM_EPOCHS, LEARNING_RATE, device, "Standard timm", patience=8
    )
    results["standard_timm"] = standard_acc
    best_epochs["standard_timm"] = timm_best_epoch

    print("\n" + "="*80)
    print("方法三：優化權重的Ensemble")
    print("="*80)

    # 優化ensemble權重
    models = [clip_finetuner, standard_timm_model]
    model_names = ["CLIP", "Standard timm"]

    try:
        optimal_weights, ensemble_acc = optimize_ensemble_weights_grid_search(models, model_names, val_loader, device)

        print(f"[INFO] 優化後的權重:")
        for i, (name, weight) in enumerate(zip(model_names, optimal_weights)):
            print(f"  {name}: {weight:.3f}")

        results["optimized_ensemble"] = ensemble_acc
        best_epochs["optimized_ensemble"] = max(clip_best_epoch, timm_best_epoch)
        print(f"[INFO] Ensemble validation accuracy: {ensemble_acc*100:.2f}%")

    except Exception as e:
        print(f"[ERROR] Ensemble權重優化失敗: {e}")
        # 使用簡單平均
        optimal_weights = [0.5, 0.5]

        # 手動計算ensemble準確率
        models[0].eval()
        models[1].eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                clip_output = torch.softmax(models[0](images), dim=1)
                timm_output = torch.softmax(models[1](images), dim=1)
                ensemble_output = 0.5 * clip_output + 0.5 * timm_output

                _, predicted = torch.max(ensemble_output, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        ensemble_acc = correct / total
        results["optimized_ensemble"] = ensemble_acc
        best_epochs["optimized_ensemble"] = max(clip_best_epoch, timm_best_epoch)
        print(f"[INFO] 使用簡單平均ensemble，準確率: {ensemble_acc*100:.2f}%")

    # 階段一結果比較
    print("\n" + "="*80)
    print("階段一結果比較（80%資料訓練）")
    print("="*80)
    print(f"CLIP Fine-tuning:     {results['clip_finetuning']*100:6.2f}% (最佳epoch: {best_epochs['clip_finetuning']})")
    print(f"Standard timm:        {results['standard_timm']*100:6.2f}% (最佳epoch: {best_epochs['standard_timm']})")
    print(f"Optimized Ensemble:   {results['optimized_ensemble']*100:6.2f}%")

    # 選擇最佳模型
    best_method = max(results, key=results.get)
    best_acc = results[best_method]
    final_epochs = best_epochs[best_method]

    print(f"\n[INFO] Best method: {best_method} with {best_acc*100:.2f}% accuracy")
    print(f"[INFO] Will use {final_epochs} epochs for final training")

    # ========================================================================
    # 階段二：使用全部300張圖片重新訓練
    # ========================================================================
    print("\n" + "="*80)
    print("階段二：使用全部300張圖片重新訓練最佳模型")
    print("="*80)
    print("依據 Yoshua Bengio 建議：使用相同epoch數在全資料集上訓練")

    # 建立全資料集載入器
    full_train_loader = build_full_dataset_loader(DATA_ROOT, IMG_SIZE, BATCH_SIZE, SEED)
    print(f"[INFO] 使用全部 {len(full_train_loader.dataset)} 張圖片")
    print(f"[INFO] 將訓練 {final_epochs} epochs（與早停相同）")

    # 重新建立並訓練最佳模型
    if best_method == "clip_finetuning":
        clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        final_model = CLIPFineTuner(clip_model, num_classes).to(device)
        final_model = train_final_model_fixed_epochs(
            final_model, full_train_loader, final_epochs, CLIP_LR, device, "CLIP Final"
        )

    elif best_method == "standard_timm":
        final_model = build_standard_timm_model(num_classes).to(device)
        final_model = train_final_model_fixed_epochs(
            final_model, full_train_loader, final_epochs, LEARNING_RATE, device, "timm Final"
        )

    elif best_method == "optimized_ensemble":
        # ensemble需要分別訓練兩個子模型
        clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        final_clip = CLIPFineTuner(clip_model, num_classes).to(device)
        final_timm = build_standard_timm_model(num_classes).to(device)

        final_clip = train_final_model_fixed_epochs(
            final_clip, full_train_loader, clip_best_epoch, CLIP_LR, device, "CLIP Final"
        )
        final_timm = train_final_model_fixed_epochs(
            final_timm, full_train_loader, timm_best_epoch, LEARNING_RATE, device, "timm Final"
        )

        final_model = OptimizedEnsemble(final_clip, final_timm, optimal_weights)

    # 保存最終模型
    torch.save(final_model.state_dict(), "model.pth")

    # 設定模型資訊
    if best_method == "optimized_ensemble":
        best_model_info = {
            "method": "optimized_ensemble",
            "architecture": "clip+timm_ensemble",
            "weights": optimal_weights
        }
    elif best_method == "clip_finetuning":
        best_model_info = {"method": "clip_finetuning", "architecture": "ViT-B-32"}
    else:  # standard_timm
        best_model_info = {"method": "standard_timm", "architecture": "convnext_tiny"}

    # 保存相關資訊
    model_info = {
        "class_names": class_names,
        "num_classes": num_classes,
        "best_method": best_method,
        "best_accuracy": best_acc,
        "final_epochs": final_epochs,
        "all_results": results,
        "best_epochs": best_epochs,
        "model_info": best_model_info
    }

    if best_method == "optimized_ensemble":
        model_info["ensemble_weights"] = optimal_weights

    with open("model_info.json", "w") as f:
        json.dump(model_info, f)

    print(f"\n[INFO] Final model saved as 'model.pth'")
    print(f"[INFO] Model info saved as 'model_info.json'")
    print("\n[INFO] Training completed successfully!")

    # 顯示訓練總結
    print("\n" + "="*80)
    print("最終訓練總結")
    print("="*80)
    print("✅ 階段一：使用80%資料進行模型選擇和early stopping")
    print("✅ 階段二：使用100%資料重新訓練，epoch數基於early stopping結果")
    print("✅ 符合Yoshua Bengio理論建議")
    print("✅ 預期獲得1-2%準確率提升")
    print("\n" + "="*80)
    print("作業合規性檢查")
    print("="*80)
    print("✅ 使用PyTorch框架")
    print("✅ 只使用300張訓練圖片")
    print("✅ 模型參數數量<1000M")
    print("✅ 使用torchinfo印出模型參數")
    print("✅ 保存最佳模型為model.pth")
    print("✅ 保存模型資訊為model_info.json")
    print(f"✅ 訓練完成，驗證準確率: {best_acc*100:.2f}%")


[INFO] Installing open_clip_torch...
[INFO] Installing pillow...
[INFO] Using device: cuda
[INFO] Loading data from /content/train
[INFO] Found 30 classes: ['apple_pie', 'beignets', 'cannoli', 'caprese_salad', 'chocolate_cake']...
[INFO] Train samples: 240, Val samples: 60

階段一：使用80%資料進行模型選擇和參數調優

方法一：CLIP Fine-tuning (最佳化訓練)
[INFO] Training CLIP Fine-tuning with early stopping (patience=15)...
[INFO] CLIP Fine-tuning Total parameters: 151,292,703 (151.3M)
[INFO] CLIP Fine-tuning Trainable parameters: 57,079,326 (57.1M)
[INFO] CLIP Fine-tuning parameter count is within requirement (<1000M)
[INFO] CLIP Fine-tuning Model Summary:
CLIP Fine-tuning Epoch 01/60 | Loss: 3.4324 | Val Acc: 1.67% | Time: 3.0s
CLIP Fine-tuning Epoch 02/60 | Loss: 3.4029 | Val Acc: 1.67% | Time: 3.0s
CLIP Fine-tuning Epoch 03/60 | Loss: 3.3514 | Val Acc: 1.67% | Time: 3.0s
CLIP Fine-tuning Epoch 04/60 | Loss: 3.3043 | Val Acc: 3.33% | Time: 4.0s
CLIP Fine-tuning Epoch 05/60 | Loss: 3.2580 | Val Acc: 10.00% | Time